# L2 Vietnamese-dish fine-grained classifier - EfficientNet Lite

Trains one **EfficientNet Lite** classifier per L1 food category. Run the
notebook once for each `TARGET` ∈ `{"noodle", "rice", "soup"}`.

### The stack

```
L1 (YOLO11s)      detects coarse category and crops the box
   │
   ▼
L2 (this notebook) classifies the specific Vietnamese dish inside the crop
```

### Dataset on Hugging Face
`WatermelonAnh/FoodClassifierL2` — one `data.zip` per L2 target
([HF dataset page](https://huggingface.co/datasets/WatermelonAnh/FoodClassifierL2)).

Each zip unpacks to standard ImageFolder layout:
`train|val|test / <class_name> / <image>.<ext>`.

### Deployment target
Final artifact is an INT8 `.tflite` with **uint8 input + uint8 output** and
class labels embedded via `tflite-support`. An FP16 `.tflite` is also produced
as a fallback. Checkpoints and exports persist to Google Drive so a Colab
disconnect never destroys progress.


In [ ]:
%%capture
%pip install -q --upgrade tensorflow-hub tflite-support huggingface_hub pillow


In [ ]:
from __future__ import annotations

import os
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from PIL import Image
from huggingface_hub import login, snapshot_download

gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise SystemExit(
        "GPU required - Runtime -> Change runtime type -> GPU (A100 recommended)."
    )

print("TF:", tf.__version__)
print("GPU:", gpus[0].name)


In [ ]:
try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    print("Outside Colab - ensure DRIVE_ROOT exists on the host filesystem.")

TOKEN = os.environ.get("HF_TOKEN", "").strip()
if TOKEN:
    login(token=TOKEN)
else:
    login()
print("Hugging Face login OK.")


In [ ]:
DRIVE_ROOT  = "/content/drive/MyDrive/FitUpNutritionL2"
RUNS_DIR    = os.path.join(DRIVE_ROOT, "runs")
EXPORTS_DIR = os.path.join(DRIVE_ROOT, "exports")

HF_DATASET_REPO = "WatermelonAnh/FoodClassifierL2"
HF_CACHE        = "/content/hf_cache"
DATA_ROOT       = "/content/l2_data"

# === The one knob to change between runs ===========================
TARGET = "noodle"           # "noodle" | "rice" | "soup"
# ===================================================================

TARGETS = {
    "noodle": dict(
        variant="lite2", imgsz=260, num_classes=12,
        tfhub_url="https://tfhub.dev/tensorflow/efficientnet/lite2/feature-vector/2",
        classes=[
            "banh_canh", "bun_bo_hue", "bun_cha", "bun_cha_ca", "bun_mam", "bun_rieu",
            "cao_lau", "hu_tieu", "mi", "mi_quang", "nui_xao_bo", "pho",
        ],
    ),
    "rice": dict(
        variant="lite1", imgsz=240, num_classes=6,
        tfhub_url="https://tfhub.dev/tensorflow/efficientnet/lite1/feature-vector/2",
        classes=[
            "chao", "com_chien", "com_chien_ga", "com_tam", "com_trang", "xoi",
        ],
    ),
    "soup": dict(
        variant="lite0", imgsz=224, num_classes=4,
        tfhub_url="https://tfhub.dev/tensorflow/efficientnet/lite0/feature-vector/2",
        classes=[
            "bo_kho", "canh", "lau", "sup_cua",
        ],
    ),
}

cfg          = TARGETS[TARGET]
DATA_DIR     = f"{DATA_ROOT}/l2_{TARGET}"
RUN_DIR      = f"{RUNS_DIR}/l2_{TARGET}"
BACKUP_DIR   = f"{RUN_DIR}/backup"
BEST_KERAS   = f"{RUN_DIR}/best.keras"
LABELS_TXT   = f"{RUN_DIR}/labels.txt"
TRAINING_LOG = f"{RUN_DIR}/training_log.csv"
FORCE_REDOWNLOAD = False

# Training hyperparameters
BATCH_SIZE       = 64
STAGE1_EPOCHS    = 8
STAGE1_LR        = 1e-3
STAGE1_PATIENCE  = 4
STAGE2_EPOCHS    = 50
STAGE2_LR        = 1e-4
STAGE2_LR_FLOOR  = 1e-6
STAGE2_PATIENCE  = 10
DROPOUT          = 0.2
CALIB_SAMPLES    = 200

os.makedirs(RUN_DIR,     exist_ok=True)
os.makedirs(EXPORTS_DIR, exist_ok=True)
print(f"Config OK. TARGET={TARGET}  variant={cfg['variant']}  "
      f"imgsz={cfg['imgsz']}  num_classes={cfg['num_classes']}")


### Helpers

The next cell is the verbatim contents of `scripts/notebook_helpers.py` from the repo. Re-run this cell after pulling helper updates.


In [ ]:
"""Pure helpers for the L2 EfficientNet Lite training notebook.

These functions are TF-free by design so they can be unit-tested locally
without a multi-GB TensorFlow install. The notebook generator embeds this
file's source verbatim into one notebook cell so the notebook stays
self-contained when run in Colab.
"""

from __future__ import annotations

import random
import shutil
import zipfile
from pathlib import Path
from typing import Callable, Iterable, Iterator

import numpy as np
from PIL import Image

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def extract_target_zip(
    hf_cache,
    data_root,
    target: str,
    force: bool = False,
) -> None:
    """Unzip hf_cache/l2_<target>/data.zip into data_root/l2_<target>/.

    Expected resulting tree:
        data_root/l2_<target>/
          train/<class_name>/<image>.<ext>
          val/<class_name>/<image>.<ext>
          test/<class_name>/<image>.<ext>

    Skip extraction when `force` is False and data_root/l2_<target>/train/
    already has at least one non-empty class subdirectory.

    Raises FileNotFoundError if the zip is missing and extraction needs to happen.
    """
    hf_cache = Path(hf_cache)
    data_root = Path(data_root)
    target_dir = data_root / f"l2_{target}"

    def already_populated() -> bool:
        train_dir = target_dir / "train"
        if not train_dir.is_dir():
            return False
        for cls_dir in train_dir.iterdir():
            if cls_dir.is_dir() and any(cls_dir.iterdir()):
                return True
        return False

    if not force and already_populated():
        print(f"[extract] {target_dir} already populated, skipping")
        return

    zip_path = hf_cache / f"l2_{target}" / "data.zip"
    if not zip_path.is_file():
        raise FileNotFoundError(f"data.zip not found at {zip_path}")

    if force and target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    print(f"[extract] {zip_path} -> {target_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)


def validate_imagefolder(
    data_dir,
    class_names: list[str],
    splits: Iterable[str] = ("train", "val", "test"),
) -> None:
    """Walk data_dir/<split>/<class>/ for every split and assert well-formedness.

    Raises ValueError on the first violation.

    Per split, asserts:
      - The split dir exists.
      - The set of class subdirs exactly matches class_names.
      - Each class subdir is non-empty.
      - Every file under each class subdir has an allowed image extension.

    On success, prints a per-split, per-class count table and the imbalance
    ratio max(count) / min(count).
    """
    data_dir = Path(data_dir)
    expected = set(class_names)

    for split in splits:
        split_dir = data_dir / split
        if not split_dir.is_dir():
            raise ValueError(f"[{split}] missing split dir: {split_dir}")

        present = {p.name for p in split_dir.iterdir() if p.is_dir()}
        missing = expected - present
        if missing:
            raise ValueError(
                f"[{split}] missing class folder(s): {sorted(missing)}"
            )
        extra = present - expected
        if extra:
            raise ValueError(
                f"[{split}] unexpected class folder(s): {sorted(extra)}"
            )

        counts: dict[str, int] = {}
        for cls in class_names:
            cls_dir = split_dir / cls
            files = [p for p in cls_dir.iterdir() if p.is_file()]
            if not files:
                raise ValueError(f"[{split}] class folder empty: {cls_dir}")
            for f in files:
                if f.suffix.lower() not in IMAGE_EXTS:
                    raise ValueError(
                        f"[{split}] non-image file: {f}  "
                        f"(allowed: {sorted(IMAGE_EXTS)})"
                    )
            counts[cls] = len(files)

        ratio = max(counts.values()) / min(counts.values())
        print(f"[{split}] total={sum(counts.values())}  imbalance(max/min)={ratio:.2f}")
        for cls in class_names:
            print(f"  {cls}: {counts[cls]}")


def compute_class_weights(data_dir, class_names: list[str]) -> dict[int, float]:
    """Return {class_index: weight} computed from the TRAIN split counts.

    weight_i = total / (num_classes * count_i)

    Balanced input yields weight ≈ 1.0 for every class. Under-represented
    classes get weight > 1; over-represented classes get weight < 1.
    Suitable for tf.keras.Model.fit(class_weight=...).
    """
    data_dir = Path(data_dir)
    train_dir = data_dir / "train"
    counts = []
    for cls in class_names:
        cls_dir = train_dir / cls
        n = sum(1 for p in cls_dir.iterdir() if p.is_file())
        if n == 0:
            raise ValueError(f"class {cls!r} has zero training samples")
        counts.append(n)
    total = sum(counts)
    nc = len(class_names)
    return {i: total / (nc * c) for i, c in enumerate(counts)}


def representative_dataset_gen(
    data_dir,
    imgsz: int,
    n: int = 200,
    seed: int = 0,
) -> Callable[[], Iterator[list[np.ndarray]]]:
    """Build a callable that yields up to `n` calibration batches.

    Each yielded item is `[ndarray of shape (1, imgsz, imgsz, 3), dtype=uint8]`
    drawn from random train images, resized via PIL.Image.LANCZOS.

    This shape matches the contract of tf.lite.TFLiteConverter.representative_dataset:
    a no-arg callable returning an iterable of input-list batches.

    If fewer than `n` train images exist, yields what's available (deterministic
    under the given seed).
    """
    data_dir = Path(data_dir)
    train_dir = data_dir / "train"
    all_images: list[Path] = []
    for cls_dir in sorted(train_dir.iterdir()):
        if not cls_dir.is_dir():
            continue
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in IMAGE_EXTS:
                all_images.append(p)

    rng = random.Random(seed)
    rng.shuffle(all_images)
    selected = all_images[:n]

    def _gen() -> Iterator[list[np.ndarray]]:
        for p in selected:
            with Image.open(p) as im:
                im = im.convert("RGB").resize((imgsz, imgsz), Image.LANCZOS)
            arr = np.asarray(im, dtype=np.uint8)[None, ...]
            yield [arr]

    return _gen


def write_labels_txt(out_path, class_names: list[str]) -> Path:
    """Write one class name per line, in the given order. Overwrites any
    existing file at out_path. Returns the path written."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n".join(class_names) + "\n")
    return out_path


### Dataset: download from HF, extract zip, validate, build tf.data


In [ ]:
if FORCE_REDOWNLOAD or not (Path(HF_CACHE) / f"l2_{TARGET}" / "data.zip").is_file():
    snapshot_download(
        repo_id="WatermelonAnh/FoodClassifierL2",
        repo_type="dataset",
        local_dir=HF_CACHE,
        allow_patterns=[f"l2_{TARGET}/*"],
    )
else:
    print(f"[download] HF cache for l2_{TARGET} already populated, skipping.")

extract_target_zip(HF_CACHE, DATA_ROOT, target=TARGET, force=FORCE_REDOWNLOAD)
print("data dir:", DATA_DIR)


### Label validation

Fails loudly if any class folder is missing, empty, or contains non-image files. Run this **before** training.


In [ ]:
validate_imagefolder(DATA_DIR, cfg["classes"])


### tf.data pipelines (train / val / test)


In [ ]:
def _make_ds(split: str, *, shuffle: bool, cache: bool):
    ds = tf.keras.utils.image_dataset_from_directory(
        f"{DATA_DIR}/{split}",
        labels="inferred",
        label_mode="int",
        class_names=cfg["classes"],          # locks label order to TARGETS
        image_size=(cfg["imgsz"], cfg["imgsz"]),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=42,
    )
    if cache:
        ds = ds.cache()
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = _make_ds("train", shuffle=True,  cache=True)
val_ds   = _make_ds("val",   shuffle=False, cache=True)
test_ds  = _make_ds("test",  shuffle=False, cache=False)

# image_dataset_from_directory yields float32 in [0,255]; cast to uint8 for the
# model's uint8 Input contract.
def _to_uint8(x, y):
    return tf.cast(x, tf.uint8), y

train_ds = train_ds.map(_to_uint8, num_parallel_calls=tf.data.AUTOTUNE)
val_ds   = val_ds.map(_to_uint8,   num_parallel_calls=tf.data.AUTOTUNE)
test_ds  = test_ds.map(_to_uint8,  num_parallel_calls=tf.data.AUTOTUNE)
print("Datasets ready.")


### Build model

EfficientNet Lite backbone from TF Hub (frozen for stage 1), with the augmentation block baked in as the first sub-block (train-time only, stripped on TFLite export). Input dtype is `uint8`; normalization to `[-1, 1]` happens inside the model.


In [ ]:
augmentation_block = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
    tf.keras.layers.RandomBrightness(0.1),
], name="augmentation")

tfhub_url = cfg["tfhub_url"]
imgsz     = cfg["imgsz"]
nc        = cfg["num_classes"]

inputs = tf.keras.Input(shape=(imgsz, imgsz, 3), dtype=tf.uint8)
x = tf.cast(inputs, tf.float32)
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)
x = augmentation_block(x)
hub_layer = hub.KerasLayer(tfhub_url, trainable=False, name="efficientnet_lite_backbone")
features = hub_layer(x)
x = tf.keras.layers.Dropout(DROPOUT)(features)
outputs = tf.keras.layers.Dense(nc, activation="softmax", name="classifier_head")(x)
model = tf.keras.Model(inputs, outputs, name=f"l2_{TARGET}_efficientnet_{cfg['variant']}")
model.summary()
